# CFP Ranker '26

## Data Collection

### Team Collection

In [2]:
%pip install requests
%pip install pandas
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [9]:
import requests
import pandas as pd
import json
import time
from datetime import datetime

# --- CONFIG ---
YEAR = 2026  # Works for any year (e.g., 2023, 2024, 2025, 2026)
FBS_TEAMS_FILE = "./espn_fbs_teams.csv"

# --- ENDPOINTS & HEADERS ---
SCOREBOARD_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"
SUMMARY_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event="

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json"
}

In [ ]:
# NOTE: This block of code extracts the list of FBS teams from ESPN's API and saves it to a CSV file.
# This is arguably more useful because it includes team IDs and conference information

url = "https://site.web.api.espn.com/apis/v2/sports/football/college-football/standings?group=80"
# 80 is the ID for the FBS (Football Bowl Subdivision) in ESPN's API

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json"
}

response = requests.get(url, headers=headers)
response.raise_for_status()

data = response.json()

def parse_standings_node(node, parent_conference="", parent_conf_id=""):
    """Recursively walks standings nodes to extract teams, conference names, and conference IDs."""
    teams = []
    
    current_name = node.get('name', '')
    current_id = str(node.get('id', ''))
    
    # Prevent top-level league/subdivision labels from overriding actual conference names/IDs
    is_root_label = current_name.lower() in ['fbs', 'ncaa - football bowl subdivision', 'football bowl subdivision']
    
    if is_root_label:
        conf_label = ""
        conf_id = ""
    else:
        conf_label = parent_conference if parent_conference else current_name
        conf_id = parent_conf_id if parent_conf_id else current_id

    # Extract teams if standings entries exist at this level
    entries = node.get('standings', {}).get('entries', [])
    for entry in entries:
        team = entry.get('team', {})
        if team:
            teams.append({
                "team_id": team.get('id'),
                "display_name": team.get('displayName'),
                "short_name": team.get('shortDisplayName'),
                "conference": conf_label,
                "conference_id": conf_id
            })

    # Recurse into sub-children (e.g. East/West divisions)
    for child in node.get('children', []):
        teams.extend(parse_standings_node(child, parent_conference=conf_label, parent_conf_id=conf_id))

    return teams

fbs_teams = parse_standings_node(data)

df = pd.DataFrame(fbs_teams)
df = df.drop_duplicates(subset=['team_id']).reset_index(drop=True)

df.to_csv("espn_fbs_teams.csv", index=False)
print(f"Successfully extracted {len(df)} FBS teams with conference IDs.")

Successfully extracted 138 FBS teams with conference IDs.


## Retrieve A Single Game

In [16]:
# --- CONFIG ---
YEAR = 2026  # Works for any year (e.g., 2023, 2024, 2025, 2026)
FBS_TEAMS_FILE = "../data/espn_fbs_teams.csv"

# --- ENDPOINTS & HEADERS ---
SCOREBOARD_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"
SUMMARY_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event="

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json"
}

params = {
    "dates": "20261017",
    "groups": "80",
    "limit": "300"
}

response = requests.get(SCOREBOARD_URL, headers=HEADERS, params=params)
target_game = None

try:
	games = response.json().get("events", [])
	if response.status_code == 200:
		for event in games:
			game_name = event.get("name", "")
			if "Alabama" in game_name and "Tennessee" in game_name:
				target_game = event
				break
except Exception as e:
	print(f"Error fetching games: {e}")
	games = []

if target_game:
	print(f"Target game found: {target_game.get('name', '')}")
else:
	print("Target game not found.")

game_id = target_game.get("id", "") if target_game else None
if game_id:
	print(f"Game ID: {game_id}")
else:
	print("Game ID not found.")

def export_boxscore_to_csv(game_id, output_filename="boxscore.csv"):
    # Base URL for the summary endpoint
    summary_url = f"https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event={game_id}"
    
    headers = HEADERS

    print(f"Fetching summary data for Game ID: {game_id}...")
    response = requests.get(summary_url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        
        # 2. Navigate to the Team Boxscore section
        teams_data = data.get("boxscore", {}).get("teams", [])
        
        if not teams_data:
            print("No box score data available yet.")
            return

        # 3. Parse the data into a list of dictionaries
        rows = []
        for team in teams_data:
            team_name = team.get("team", {}).get("displayName", "Unknown")
            stats = team.get("statistics", [])
            
            # Create a dictionary for the row, starting with the team name
            team_row = {"Team": team_name}
            
            # Loop through each stat (e.g., 'totalYards', 'firstDowns') and add it to the row
            for stat in stats:
                stat_name = stat.get("label", stat.get("name"))
                stat_value = stat.get("displayValue")
                team_row[stat_name] = stat_value
                
            rows.append(team_row)

        # 4. Use Pandas to convert the rows to a DataFrame and export to CSV
        df = pd.DataFrame(rows)
        
        # Optional: Transpose the DataFrame if you want the stats as rows instead of columns
        df = df.set_index("Team").T
        df.reset_index(inplace=True)
        df.rename(columns={"index": "Stat Category"}, inplace=True)
        
        df.to_csv(output_filename, index=False)
        print(f"\nSuccess! Data exported to {output_filename}")
        print(df.head())
        
    else:
        print(f"Failed to retrieve data. Status Code: {response.status_code}")

# Use the game ID we found in the previous step
TARGET_GAME_ID = game_id # Replace with your dynamic Game ID
export_boxscore_to_csv(TARGET_GAME_ID, f"Tennessee_Alabama_Boxscore_{TARGET_GAME_ID}.csv")

def save_full_game_json(game_id, output_filename="full_game_data.json"):
    summary_url = f"https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event={game_id}"
    
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json"
    }

    print(f"Fetching full summary JSON for Game ID: {game_id}...")
    response = requests.get(summary_url, headers=headers)

    if response.status_code == 200:
        # Parse the JSON response
        data = response.json()
        
        # Open a new file in write mode and dump the JSON into it
        # indent=4 makes the file highly readable (pretty-printed)
        with open(output_filename, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=4)
            
        print(f"Success! The full JSON payload has been saved to '{output_filename}'.")
    else:
        print(f"Failed to retrieve data. Status Code: {response.status_code}")

# The Game ID for your specific matchup
TARGET_GAME_ID = game_id

save_full_game_json(TARGET_GAME_ID, "Tennessee_Alabama_Full_Summary.json")

Target game found: Alabama Crimson Tide at Tennessee Volunteers
Game ID: 401856723
Fetching summary data for Game ID: 401856723...

Success! Data exported to Tennessee_Alabama_Boxscore_401856723.csv
Team        Stat Category Alabama Crimson Tide Tennessee Volunteers
0               1st Downs                   12                   20
1     3rd down efficiency                 2-13                 6-15
2     4th down efficiency                  1-3                  1-1
3             Total Yards                  201                  347
4                 Passing                  147                  197
Fetching full summary JSON for Game ID: 401856723...
Success! The full JSON payload has been saved to 'Tennessee_Alabama_Full_Summary.json'.


In [ ]:
def discover_logged_statistics(input_filename="Tennessee_Alabama_Full_Summary.json", output_filename="logged_statistics.txt"):
    num_team_stats = 0
    num_player_stats = 0
    # Load the JSON data
    with open(input_filename, "r", encoding="utf-8") as file:
        data = json.load(file)
        
    # Open the output text file in write mode
    with open(output_filename, "w", encoding="utf-8") as out_file:
        out_file.write("=== DISCOVERING LOGGED STATISTICS ===\n\n")
        
        boxscore = data.get("boxscore", {})
        
        # 1. Extract Team-Level Statistics
        out_file.write("--- TEAM STATISTICS TRACKED ---\n")
        teams = boxscore.get("teams", [])
        if teams:
            team_stats = teams[0].get("statistics", [])
            for stat in team_stats:
                stat_name = stat.get("name", "Unknown key")
                stat_label = stat.get("label", "Unknown label")
                out_file.write(f"- {stat_label} (Key: {stat_name})\n")
                num_team_stats += 1
        else:
            out_file.write("No team statistics found.\n")
            
        # 2. Extract Individual Player Stat Categories
        out_file.write("\n--- INDIVIDUAL PLAYER STAT CATEGORIES ---\n")
        players = boxscore.get("players", [])
        if players:
            player_categories = players[0].get("statistics", [])
            for category in player_categories:
                cat_name = category.get("name", "Unknown")
                out_file.write(f"\n[{cat_name.upper()}]\n")
                
                keys = category.get("keys", [])
                labels = category.get("labels", [])
                
                for key, label in zip(keys, labels):
                    out_file.write(f"  - {label} (Key: {key})\n")
                    num_player_stats += 1
        else:
            out_file.write("No player statistics found.\n")

    # Print a single confirmation message to the console
    print(f"Success! The full list of statistics has been saved to '{output_filename}'.")
    print(f"Total team statistics tracked: {num_team_stats}")
    print(f"Total individual player stat categories: {num_player_stats}")
    
discover_logged_statistics("Tennessee_Alabama_Full_Summary.json")

Success! The full list of statistics has been saved to 'logged_statistics.txt'.
Total team statistics tracked: 15
Total individual player stat categories: 50


# Moving Forward

## Plan A

Find the missing values, document that, and determine what ESPN stat loggers deam as most important

## Plan B

If there are no missing values, find ways to use the data to make conjecture about the strength of teams based on their game-by-game stats.  Could be worth trying to find that play-by-play endpoint

### Retrieve All The Games For The 2026 Season

In [9]:
# Retrieve all FBS games for the 2026 season and save them to a CSV file

start_date = pd.Timestamp("2026-08-01")
end_date = pd.Timestamp("2027-02-01")

rows = []
chunk_start = start_date

while chunk_start <= end_date:
	response = requests.get(
		SCOREBOARD_URL,
		headers=HEADERS,
		params={
			"dates": f"{chunk_start:%Y%m%d}",
			"groups": "80",
			"limit": "500",
		},
		timeout=30,
	)
	response.raise_for_status()

	for event in response.json().get("events", []):
		competition = (event.get("competitions") or [{}])[0]
		competitors = competition.get("competitors", [])
		home = next((team for team in competitors if team.get("homeAway") == "home"), {})
		away = next((team for team in competitors if team.get("homeAway") == "away"), {})

		rows.append({
			"event_id": event.get("id"),
			"date": event.get("date"),
			"name": event.get("name"),
			"week": event.get("week", {}).get("number"),
			"home_team": home.get("team", {}).get("displayName"),
			"home_team_id": home.get("team", {}).get("id"),
			"home_score": home.get("score"),
			"away_team": away.get("team", {}).get("displayName"),
			"away_team_id": away.get("team", {}).get("id"),
			"away_score": away.get("score"),
			"status": event.get("status", {}).get("type", {}).get("description"),
			"venue": competition.get("venue", {}).get("fullName"),
			"neutral_site": competition.get("neutralSite"),
		})

	chunk_start += pd.Timedelta(days=1)
	time.sleep(0.2)

season_games_df = (
	pd.DataFrame(rows)
	.drop_duplicates(subset="event_id")
	.sort_values("date")
	.reset_index(drop=True)
)

output_file = "espn_2026_fbs_season_games.csv"
season_games_df.to_csv(output_file, index=False)

print(f"Retrieved {len(season_games_df)} games and saved them to {output_file}.")
season_games_df.head()

Retrieved 950 games and saved them to espn_2026_fbs_season_games.csv.


,event_id,date,name,week,home_team,home_team_id,home_score,away_team,away_team_id,away_score,status,venue,neutral_site
0,401856766,2026-08-29T16:00Z,North Carolina Tar Heels at TCU Horned Frogs,1,TCU Horned Frogs,2628,10,North Carolina Tar Heels,153,15,Final,Aviva Stadium,True
1,401864494,2026-08-29T19:00Z,San José State Spartans at USC Trojans,1,USC Trojans,30,42,San José State Spartans,23,26,Final,Los Angeles Memorial Coliseum,False
2,401858202,2026-08-29T19:30Z,NC State Wolfpack at Virginia Cavaliers,1,Virginia Cavaliers,258,34,NC State Wolfpack,152,8,Final,Scott Stadium,False
3,401864577,2026-08-29T21:30Z,Jacksonville State Gamecocks at North Dakota S...,1,North Dakota State Bison,2449,33,Jacksonville State Gamecocks,55,7,Final,Fargodome,False
4,401866408,2026-08-29T22:30Z,Sacramento State Hornets at Eastern Michigan E...,1,Eastern Michigan Eagles,2199,28,Sacramento State Hornets,16,17,Final,Rynearson Stadium,False


### Check for Missing Stats

In [10]:
season_games_df = pd.read_csv("espn_2026_fbs_season_games.csv")

# For every game, we want to check if there are any missing values among the 65 statistical categories 
# that ESPN tracks for each game. If any are missing, we will log the game ID and the missing categories to a text file.
with open("missing_stats.log", "w") as f:
	for index, row in season_games_df.iterrows():
		if row["status"] == "Final":
			game_id = row["event_id"]
			summary_url = f"https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event={game_id}"
			
			response = requests.get(summary_url, headers=HEADERS)
			if response.status_code != 200:
				f.write(f"Game ID {game_id}: Failed to retrieve data (Status Code: {response.status_code})\n")
				continue
			
			data = response.json()
			boxscore = data.get("boxscore", {})
			
			# Check team statistics
			teams = boxscore.get("teams", [])
			for team in teams:
				stats = team.get("statistics", [])
				for stat in stats:
					if stat.get("displayValue") is None:
						f.write(f"Game ID {game_id}: Missing team stat '{stat.get('label')}'\n")
			
			# Check player statistics
			players = boxscore.get("players", [])
			for player in players:
				categories = player.get("statistics", [])
				for category in categories:
					keys = category.get("keys", [])
					labels = category.get("labels", [])
					values = category.get("values", [])
					for key, label, value in zip(keys, labels, values):
						if value is None:
							f.write(f"Game ID {game_id}: Missing player stat '{label}' (Key: {key})\n")

with open("missing_stats.log", "r") as f:
	content = f.read().strip()
	if not content:
		print("No missing statistics found for any game.")
	else:
		print("Missing statistics were found. Please check 'missing_stats.log' for details.")

No missing statistics found for any game.


## Execute Plan B

An interesting thing to note is that the team stats provided are mostly offensive statistics.  These can be used to make conjecture about team defenses though.  If you were to aggregate a team’s opponents’ offensive stats, you could use that to make conjecture about their defense.

Additionally, you could compare how that opposing team did in that game versus in other games to make conjecture about a team's defense.

*i.e. Alabama had a high flying offense in their previous games but was shut down by both Georgia’s and Tennessee’s defenses.  Statistically, that shows across various categories that are measured in the API.  We can use that to make conjecture about Alabama’s offense, but also Georgia’s and Tennessee’s defenses.*

That information can be used to neutralize statistical outliers or upset games in a ranking algorithm.

In [11]:
# As an example, let's retrieve the box scores for all games played by Tennessee and their opponents.
# We can store this data in a CSV file for further analysis.
import requests
import pandas as pd
import json
import time
from datetime import datetime

# --- CONFIG ---
YEAR = 2026  # Works for any year (e.g., 2023, 2024, 2025, 2026)
FBS_TEAMS_FILE = "./espn_fbs_teams.csv"

# --- ENDPOINTS & HEADERS ---
SCOREBOARD_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/scoreboard"
SUMMARY_URL = "https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event="

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json"
}

season_games_df = pd.read_csv("espn_2026_fbs_season_games.csv")

# Rewrite this to store the data in a dataframe
def retrieve_boxscore(game_id):
	summary_url = f"https://site.web.api.espn.com/apis/site/v2/sports/football/college-football/summary?event={game_id}"
	
	response = requests.get(summary_url, headers=HEADERS)
	if response.status_code != 200:
		print(f"Failed to retrieve data for Game ID {game_id}. Status Code: {response.status_code}")
		return None
	
	data = response.json()
	boxscore = data.get("boxscore", {})
	
	# Extract team statistics
	teams_data = boxscore.get("teams", [])
	team_stats_list = []
	
	for team in teams_data:
		team_name = team.get("team", {}).get("displayName", "Unknown")
		stats = team.get("statistics", [])
		
		for stat in stats:
			stat_name = stat.get("name", stat.get("name"))
			stat_value = stat.get("value")
			if stat_value == '-':
				stat_value = stat.get("displayValue")
			if stat_name == "fourthDownEff" and stat_value == "0-0":
				stat_value = None
			team_stats_list.append({
				"Game ID": game_id,
				"Team": team_name,
				"Stat Category": stat_name,
				"Stat Value": stat_value
			})
	
	return pd.DataFrame(team_stats_list)
	
fbs_teams_df = pd.read_csv(FBS_TEAMS_FILE)
tn_team_id = fbs_teams_df.loc[fbs_teams_df['display_name'] == 'Tennessee Volunteers', 'team_id'].values[0]

season_games_df = pd.read_csv("espn_2026_fbs_season_games.csv")
# Filter for games where Tennessee is either the home or away team
tn_games_df = season_games_df[((season_games_df['home_team_id'] == tn_team_id) | (season_games_df['away_team_id'] == tn_team_id)) & (season_games_df['status'] == 'Final')]

for index, row in tn_games_df.iterrows():
	game_id = row['event_id']
	boxscore_df = retrieve_boxscore(game_id)
	if boxscore_df is not None:
		output_filename = f"./data/Tennessee_Game_{game_id}_Boxscore.csv"
		boxscore_df.to_csv(output_filename, index=False)

In [12]:
# Now, let's retrieve the box scores for all games played by Tennessee's opponents and store them in a CSV file for further analysis.
opponent_team_ids = set(tn_games_df['home_team_id'].tolist() + tn_games_df['away_team_id'].tolist())

for opponent_id in opponent_team_ids:
	opponent_games_df = season_games_df[((season_games_df['home_team_id'] == opponent_id) | (season_games_df['away_team_id'] == opponent_id)) & (season_games_df['status'] == 'Final')]
	
	for index, row in opponent_games_df.iterrows():
		game_id = row['event_id']
		boxscore_df = retrieve_boxscore(game_id)
		if boxscore_df is not None:
			output_filename = f"./data/Opponent_{opponent_id}_Game_{game_id}_Boxscore.csv"
			boxscore_df.to_csv(output_filename, index=False)


In [7]:
# Use the collected box score data to analyze Tennessee's defensive performance against their opponents in the 2026 season.
# We can aggregate the statistics and look for trends or patterns that may indicate strengths or weaknesses in Tennessee's defense.

def calculate_defensive_performance(games_df, team_name):
	opponent_stats_list = []

	for _, row in games_df.iterrows():
		boxscore_df = retrieve_boxscore(row["event_id"])

		if boxscore_df is not None and not boxscore_df.empty:
			opponent_stats = boxscore_df[
				boxscore_df["Team"] != team_name
			].copy()
			opponent_stats_list.append(opponent_stats)

	if not opponent_stats_list:
		return pd.DataFrame(columns=["Stat Category", "Stat Value"])

	defensive_stats = pd.concat(opponent_stats_list, ignore_index=True)
	defensive_stats["Stat Value"] = pd.to_numeric(
		defensive_stats["Stat Value"], errors="coerce"
	)

	excluded = defensive_stats.loc[
		defensive_stats["Stat Value"].isna(), "Stat Category"
	].unique()

	aggregated = (
		defensive_stats.dropna(subset=["Stat Value"])
		.groupby("Stat Category", as_index=False)["Stat Value"]
		.mean()
	)

	# Rename all stat categories to "avg_<stat_category>" to indicate that these are average values across all games
	aggregated["Stat Category"] = aggregated["Stat Category"].apply(lambda x: f"avg_{x}_against")

	if len(excluded):
		print("Excluded nonnumeric stats:", ", ".join(excluded))

	return aggregated


calculated_defensive_stats = calculate_defensive_performance(tn_games_df, "Tennessee Volunteers")
calculated_defensive_stats

Excluded nonnumeric stats: completionAttempts, totalPenaltiesYards, fourthDownEff


,Stat Category,Stat Value
0,avg_firstDowns_against,16.500000
1,avg_fourthDownEff_against,0.200000
2,avg_fumblesLost_against,0.750000
3,avg_interceptions_against,1.000000
4,avg_netPassingYards_against,165.625000
5,avg_possessionTime_against,1718.500000
6,avg_rushingAttempts_against,32.000000
7,avg_rushingYards_against,122.750000
8,avg_thirdDownEff_against,0.314646
9,avg_totalYards_against,288.375000


## Store Season Data in SQLite

The next cells create a local relational database and load each completed FBS game in `season_games_df`. `Box_Scores` stores one row per game, team, and statistic, keeping the statistic catalog flexible.

In [3]:
import sqlite3
import pandas as pd

YEAR = 2026

DB_FILE = f"espn_cfb_{YEAR}.db"
conn = sqlite3.connect(DB_FILE)
conn.execute("PRAGMA foreign_keys = ON")

conn.executescript("""
CREATE TABLE IF NOT EXISTS Teams (
    team_id TEXT PRIMARY KEY,
    display_name TEXT NOT NULL,
    short_name TEXT,
    conference TEXT,
    conference_id TEXT
);
CREATE TABLE IF NOT EXISTS Games (
    game_id TEXT PRIMARY KEY,
    game_date TEXT,
    name TEXT,
    week INTEGER,
    home_team_id TEXT REFERENCES Teams(team_id),
    away_team_id TEXT REFERENCES Teams(team_id),
    home_score INTEGER,
    away_score INTEGER,
    status TEXT,
    venue TEXT,
    neutral_site INTEGER
);
CREATE TABLE IF NOT EXISTS Box_Scores (
    game_id TEXT NOT NULL REFERENCES Games(game_id),
    team_id TEXT NOT NULL REFERENCES Teams(team_id),
    stat_name TEXT NOT NULL,
    stat_label TEXT,
    stat_value REAL,
    display_value TEXT,
    PRIMARY KEY (game_id, team_id, stat_name)
);
CREATE INDEX IF NOT EXISTS idx_box_scores_team ON Box_Scores(team_id);
CREATE INDEX IF NOT EXISTS idx_games_date ON Games(game_date);
""")
conn.commit()
print(f"SQLite database ready: {DB_FILE}")

SQLite database ready: espn_cfb_2026.db


In [8]:
# The Team Collection cell creates this file; alternatively use its in-memory DataFrame.
import os

FBS_TEAMS_FILE = f"espn_fbs_teams.csv"

if os.path.exists(FBS_TEAMS_FILE):
    teams_df = pd.read_csv(FBS_TEAMS_FILE, dtype={"team_id": str})
elif "df" in globals() and {"team_id", "display_name"}.issubset(df.columns):
    teams_df = df.copy()
else:
    raise FileNotFoundError("Run the Team Collection cell first to create the FBS team list.")

teams_df = teams_df.drop_duplicates(subset=["team_id"]).copy()
teams_df["team_id"] = teams_df["team_id"].astype(str)
if len(teams_df) != 138:
    print(f"Expected 138 FBS teams; found {len(teams_df)}. Loading the available team list.")

def optional(row, column):
    value = row.get(column)
    return None if pd.isna(value) else str(value)

team_rows = [(str(row["team_id"]), row["display_name"], row.get("short_name"),
              row.get("conference"), optional(row, "conference_id"))
             for _, row in teams_df.iterrows()]
conn.executemany("""INSERT INTO Teams (team_id, display_name, short_name, conference, conference_id)
VALUES (?, ?, ?, ?, ?) ON CONFLICT(team_id) DO UPDATE SET
 display_name=excluded.display_name, short_name=excluded.short_name,
 conference=excluded.conference, conference_id=excluded.conference_id""", team_rows)
conn.commit()
print(f"Loaded {len(team_rows)} teams.")

Loaded 138 teams.


In [21]:
import requests

SUMMARY_URL = "https://site.api.espn.com/apis/site/v2/sports/football/college-football/summary?event="
HEADERS = {"User-Agent": "Mozilla/5.0"}

# Fetch each completed game's summary once; both teams' statistics are in each response.
season_games_df = pd.read_csv("espn_2026_fbs_season_games.csv")

games_to_load = season_games_df.drop_duplicates(subset="event_id").copy()
games_to_load = games_to_load[games_to_load["status"].eq("Final")]

game_insert = """
    INSERT INTO Games (game_id, game_date, name, week, home_team_id, away_team_id, home_score, away_score, status, venue, neutral_site) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ON CONFLICT(game_id) DO UPDATE SET game_date=excluded.game_date, name=excluded.name, week=excluded.week,
        home_team_id=excluded.home_team_id, away_team_id=excluded.away_team_id, home_score=excluded.home_score,
        away_score=excluded.away_score, status=excluded.status, venue=excluded.venue, neutral_site=excluded.neutral_site
"""
boxscore_insert = """
    INSERT INTO Box_Scores (game_id, team_id, stat_name, stat_label, stat_value, display_value) VALUES (?, ?, ?, ?, ?, ?)
    ON CONFLICT(game_id, team_id, stat_name) DO UPDATE SET stat_label=excluded.stat_label, stat_value=excluded.stat_value, display_value=excluded.display_value
"""

def integer_or_none(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass
    try:
        return int(value)
    except (TypeError, ValueError):
        return None

loaded_games = 0
failed_games = []
for _, game in games_to_load.iterrows():
    game_id = str(game["event_id"])
    try:
        response = requests.get(f"{SUMMARY_URL}{game_id}", headers=HEADERS, timeout=30)
        response.raise_for_status()
        summary = response.json()
        team_stats = summary.get("boxscore", {}).get("teams", [])
        if not team_stats:
            continue

        # 1) Make sure every team referenced by the game exists FIRST
        for team in team_stats:
            info = team.get("team", {})
            if info.get("id") is not None:
                conn.execute(
                    "INSERT OR IGNORE INTO Teams (team_id, display_name) VALUES (?, ?)",
                    (str(info["id"]), info.get("displayName", "Unknown")),
                )
        # Also cover home/away IDs from the schedule data, in case they differ
        for col, name_col in (("home_team_id", "home_team"), ("away_team_id", "away_team")):
            tid = game.get(col)
            if pd.notna(tid):
                conn.execute(
                    "INSERT OR IGNORE INTO Teams (team_id, display_name) VALUES (?, ?)",
                    (str(tid), str(game.get(name_col) or "Unknown")),
                )

        # 2) Now insert the game
        conn.execute(
            game_insert,
            (
                game_id,
                game.get("date"),
                game.get("name"),
                integer_or_none(game.get("week")),
                str(game["home_team_id"]) if pd.notna(game.get("home_team_id")) else None,
                str(game["away_team_id"]) if pd.notna(game.get("away_team_id")) else None,
                integer_or_none(game.get("home_score")),
                integer_or_none(game.get("away_score")),
                game.get("status"),
                game.get("venue"),
                int(bool(game.get("neutral_site"))) if pd.notna(game.get("neutral_site")) else None,
            ),
        )

        stat_rows = []
        team_stats = summary.get("boxscore", {}).get("teams", [])
        for team in team_stats:
            team_info = team.get("team", {})
            team_id = team_info.get("id")
            if team_id is None:
                continue
            team_id = str(team_id)
            # Includes FCS opponents when they appear in an FBS game, satisfying the FK.
            conn.execute(
                "INSERT OR IGNORE INTO Teams (team_id, display_name) VALUES (?, ?)",
                (team_id, team_info.get("displayName", "Unknown")),
            )
            for stat in team.get("statistics", []):
                stat_name = stat.get("name") or stat.get("label")
                if not stat_name:
                    continue
                try:
                    numeric_value = float(stat["value"]) if stat.get("value") is not None else None
                except (TypeError, ValueError):
                    numeric_value = None
                stat_rows.append(
                    (
                        game_id,
                        team_id,
                        str(stat_name),
                        stat.get("label"),
                        numeric_value,
                        stat.get("displayValue"),
                    )
                )

        conn.executemany(boxscore_insert, stat_rows)
        conn.commit()
        loaded_games += 1
        time.sleep(0.15)
    except requests.RequestException as exc:
        conn.rollback()
        failed_games.append((game_id, str(exc)))
    except Exception:
        conn.rollback()
        raise

print(f"Loaded box scores for {loaded_games} games; {len(failed_games)} requests failed.")
if failed_games:
    print("Failed game IDs:", [game_id for game_id, _ in failed_games])
print(
    "Database counts:",
    conn.execute(
        "SELECT (SELECT COUNT(*) FROM Teams), (SELECT COUNT(*) FROM Games), (SELECT COUNT(*) FROM Box_Scores)"
    ).fetchone(),
)

Loaded box scores for 684 games; 1 requests failed.
Failed game IDs: ['401871076']
Database counts: (236, 685, 20550)


In [22]:
final_ids = set(season_games_df.loc[season_games_df["status"] == "Final", "event_id"].astype(str))
loaded_ids = {r[0] for r in conn.execute("SELECT game_id FROM Games")}
print("Final games not loaded:", len(final_ids - loaded_ids))

print("FBS teams with no games:", conn.execute("""
    SELECT t.display_name FROM Teams t
    WHERE t.conference IS NOT NULL
      AND NOT EXISTS (SELECT 1 FROM Games g WHERE t.team_id IN (g.home_team_id, g.away_team_id))
""").fetchall())

print("FK violations:", conn.execute("PRAGMA foreign_key_check").fetchall())

Final games not loaded: 0
FBS teams with no games: []
FK violations: []


In [23]:
pd.read_sql_query("""
SELECT g.week, g.game_date, h.display_name AS home, g.home_score,
       a.display_name AS away, g.away_score
FROM Games g
JOIN Teams h ON h.team_id = g.home_team_id
JOIN Teams a ON a.team_id = g.away_team_id
ORDER BY g.game_date DESC
LIMIT 10
""", conn)

,week,game_date,home,home_score,away,away_score
0,10,2026-11-08T03:30Z,UNLV Rebels,35,Wyoming Cowboys,41
1,10,2026-11-08T03:30Z,Oregon State Beavers,21,Texas State Bobcats,42
2,10,2026-11-08T02:30Z,Texas Tech Red Raiders,53,West Virginia Mountaineers,48
3,10,2026-11-08T02:30Z,UTEP Miners,14,Hawai'i Rainbow Warriors,17
4,10,2026-11-08T02:30Z,Utah State Aggies,17,Fresno State Bulldogs,10
5,10,2026-11-08T01:00Z,New Mexico State Aggies,14,Liberty Flames,30
6,10,2026-11-08T00:45Z,Florida Gators,7,Oklahoma Sooners,23
7,10,2026-11-08T00:30Z,Ole Miss Rebels,7,Georgia Bulldogs,42
8,10,2026-11-08T00:30Z,Northwestern Wildcats,10,Iowa Hawkeyes,24
9,10,2026-11-08T00:30Z,Army Black Knights,21,Air Force Falcons,24


In [24]:
pd.read_sql_query("SELECT DISTINCT stat_name, stat_label FROM Box_Scores ORDER BY stat_name", conn)

pd.read_sql_query("""
SELECT t.display_name, g.name AS game, b.stat_value AS rushing_yards
FROM Box_Scores b
JOIN Teams t ON t.team_id = b.team_id
JOIN Games g ON g.game_id = b.game_id
WHERE b.stat_name = 'rushingYards'
ORDER BY b.stat_value DESC
LIMIT 10
""", conn)

,display_name,game,rushing_yards
0,Army Black Knights,Bryant Bulldogs at Army Black Knights,569.0
1,Navy Midshipmen,Towson Tigers at Navy Midshipmen,506.0
2,Texas State Bobcats,Colorado State Rams at Texas State Bobcats,503.0
3,Rice Owls,Rice Owls at Florida Atlantic Owls,499.0
4,Kansas Jayhawks,Middle Tennessee Blue Raiders at Kansas Jayhawks,480.0
5,Army Black Knights,Florida Atlantic Owls at Army Black Knights,479.0
6,Rice Owls,Houston Christian Huskies at Rice Owls,458.0
7,Air Force Falcons,Air Force Falcons at Northern Illinois Huskies,439.0
8,Troy Trojans,Southern Miss Golden Eagles at Troy Trojans,437.0
9,Air Force Falcons,Duquesne Dukes at Air Force Falcons,431.0


In [26]:
conn.executescript("""
CREATE TABLE IF NOT EXISTS Defense_Stats (
    game_id       TEXT NOT NULL REFERENCES Games(game_id),
    team_id       TEXT NOT NULL REFERENCES Teams(team_id),   -- the defense
    opponent_id   TEXT NOT NULL REFERENCES Teams(team_id),   -- the offense it faced
    stat_name     TEXT NOT NULL,
    stat_label    TEXT,
    stat_value    REAL,
    display_value TEXT,
    PRIMARY KEY (game_id, team_id, stat_name)
);
CREATE INDEX IF NOT EXISTS idx_defense_team ON Defense_Stats(team_id);
""")
conn.commit()

In [27]:
import pandas as pd

box   = pd.read_sql_query("SELECT * FROM Box_Scores", conn, dtype={"team_id": str, "game_id": str})
games = pd.read_sql_query("SELECT * FROM Games", conn, dtype={"home_team_id": str, "away_team_id": str, "game_id": str})

# Pair each team with its opponent in the same game
teams = box[["game_id", "team_id"]].drop_duplicates()
pairs = teams.merge(teams, on="game_id", suffixes=("_off", "_def"))
pairs = pairs[pairs.team_id_off != pairs.team_id_def]

# Opponent offensive stats become this team's "allowed" stats
allowed = box.merge(pairs, left_on=["game_id", "team_id"], right_on=["game_id", "team_id_off"])
allowed = allowed.rename(columns={"team_id_def": "def_team", "team_id_off": "opponent_id"})
allowed = pd.DataFrame({
    "game_id":       allowed["game_id"],
    "team_id":       allowed["def_team"],
    "opponent_id":   allowed["opponent_id"],
    "stat_name":     "opp_" + allowed["stat_name"],      # prefix keeps meaning clear
    "stat_label":    "Opp " + allowed["stat_label"].fillna(""),
    "stat_value":    allowed["stat_value"],
    "display_value": allowed["display_value"],
})

# Derived stat: points allowed (and scored), from Games
home = games[["game_id", "home_team_id", "away_team_id", "away_score"]].set_axis(
    ["game_id", "team_id", "opponent_id", "stat_value"], axis=1)
away = games[["game_id", "away_team_id", "home_team_id", "home_score"]].set_axis(
    ["game_id", "team_id", "opponent_id", "stat_value"], axis=1)
pts = pd.concat([home, away], ignore_index=True)
pts["stat_name"], pts["stat_label"] = "points_allowed", "Points Allowed"
pts["display_value"] = pts["stat_value"].astype("Int64").astype(str)

defense = pd.concat([allowed, pts], ignore_index=True)
defense = defense[defense["game_id"].isin(box["game_id"])]   # only games with box scores

In [28]:
def split_compound(df, stat_name, first, second):
    sub = df[df["stat_name"] == stat_name].copy()
    parts = sub["display_value"].str.extract(r"^\s*(\d+)\s*-\s*(\d+)\s*$").astype(float)
    out = []
    for col, new_name in zip(parts.columns, (first, second)):
        o = sub.copy()
        o["stat_name"], o["stat_label"], o["stat_value"] = new_name, new_name, parts[col]
        out.append(o)
    return pd.concat(out)

# e.g. opp_sacksTaken -> sacks made by this defense + yards lost
defense = pd.concat([defense, split_compound(defense, "opp_sacksTaken", "sacks", "sack_yards")])


In [29]:
rows = defense.astype(object).where(defense.notna(), None)
rows = list(rows[["game_id","team_id","opponent_id","stat_name",
                  "stat_label","stat_value","display_value"]].itertuples(index=False, name=None))

conn.executemany("""
INSERT INTO Defense_Stats (game_id, team_id, opponent_id, stat_name, stat_label, stat_value, display_value)
VALUES (?, ?, ?, ?, ?, ?, ?)
ON CONFLICT(game_id, team_id, stat_name) DO UPDATE SET
  opponent_id=excluded.opponent_id, stat_label=excluded.stat_label,
  stat_value=excluded.stat_value, display_value=excluded.display_value
""", rows)
conn.commit()
print(conn.execute("SELECT COUNT(*) FROM Defense_Stats").fetchone())

(21920,)


In [30]:
pd.read_sql_query("""
SELECT stat_name, stat_label, COUNT(*) AS rows_, COUNT(stat_value) AS numeric_rows
FROM Defense_Stats
GROUP BY stat_name, stat_label
ORDER BY stat_name
""", conn)

,stat_name,stat_label,rows_,numeric_rows
0,opp_completionAttempts,Opp Comp/Att,1370,0
1,opp_firstDowns,Opp 1st Downs,1370,1370
2,opp_fourthDownEff,Opp 4th down efficiency,1370,1141
3,opp_fumblesLost,Opp Fumbles lost,1370,1370
4,opp_interceptions,Opp Interceptions thrown,1370,1370
5,opp_netPassingYards,Opp Passing,1370,1370
6,opp_possessionTime,Opp Possession,1370,1370
7,opp_rushingAttempts,Opp Rushing Attempts,1370,1370
8,opp_rushingYards,Opp Rushing,1370,1370
9,opp_thirdDownEff,Opp 3rd down efficiency,1370,1370
